# Real-World Prompting

Prompt engineering is not just about asking a model a question. It is about designing a clear task, providing the right context, and defining what a useful response should look like.

In this notebook, we will move from simple prompts to a **more structured approach for a real-world task: analyzing customer reviews**.

We will progressively improve a prompt by adding:  

- a clear role and objective  
- task-specific instructions  
- reusable prompt templates   
- explicit output requirements  
- examples that demonstrate the expected behavior  

We will then test the prompt with different reviews and explore how prompt design affects the consistency and usefulness of the results.  

The goal is not to find a single "perfect" prompt, but to learn a repeatable process for designing, testing, and improving prompts. In real-world applications, a well-structured prompt can make the difference between vague and unreliable answers and a useful, repeatable workflow. 

## Learning Objectives

By the end of this notebook, you will be able to:

- identify the key components of a well-structured prompt
- separate task instructions from input data
- create reusable prompt templates
- define a structured output format
- use examples to guide model behavior
- test a prompt with different inputs
- evaluate whether model outputs meet the requirements of a task

##  Setup  

Before making our first request to Claude, we need to prepare the Python environment and initialize the Anthropic client.  It begins by importing the client, loading environment variables, and selecting a model, which is the typical setup needed for any Anthropic API usage. 


### Import SDK  
 The anthropic package provides the Python SDK for interacting with Claude through the Anthropic API. 

In [1]:
# Import the Anthropic SDK client.
from anthropic import Anthropic

### Load credentials  
  
This repository uses environment variables to keep the Anthropic API key separate from the code. The provided helper function (`helper.py`) loads the variables from the local `.env` file (see `.env_example` file). 

In [2]:
# Load environment variables such as ANTHROPIC_API_KEY
from helper import load_env
load_env()

### Initialize client  

Once the environment variables have been loaded, create an Anthropic client. The client will use the ANTHROPIC_API_KEY environment variable when making API requests.

In [3]:
# Initialize the Anthropic client
client = Anthropic()


# Example model configuration.
MODEL_NAME="claude-sonnet-5"  

##  From Simple Prompts to Structured Prompts

A prompt does not need to be complicated to be useful. However, as a task becomes more specific, a simple instruction may not provide enough guidance for the model to produce a consistent and useful result.

We will start with a **simple prompt** and then improve it by adding structure.  


### A Simple Prompt

Consider the following request, which is an example of a "consumer prompt", demostrating a casual user request:  

In [4]:
consumer_prompt = "Help me brainstorm ideas for a talk on AI and education"

This prompt gives Claude a general goal, but leaves several questions unanswered:  

- Who is the audience?  
- What type of ideas are we looking for?  
- How many ideas should Claude provide?  
- How should the ideas be organized?  
- Are there any constraints or requirements?  

For an informal brainstorming task, this may be perfectly adequate. However, when we build an application, we often need the model to produce a more predictable result. 

The goal is therefore not to make every prompt longer. Instead, we want to provide the information that is important for the task.   

For a casual conversation, this may be perfectly adequate. Claude can make reasonable assumptions and provide a useful answer.

For an application that needs **consistent and predictable results**, however, we usually need to be more explicit.

### A Structured Prompt

A more structured prompt explicitly defines the key elements of the task.  

A structured prompt defines the task more clearly by separating the main components of the request.

A useful starting point is: 

```text
Task  
Context  
Instructions  
Output format  
Constraints
```

For example:

In [5]:
structured_prompt = """ You are helping prepare a presentation about AI and education. 

Task: Generate five possible topics for a 20-minute presentation. 

Context:
The audience consists of university educators who have a basic 
understanding of artificial intelligence.   

Instructions: 
- Focus on practical applications of AI in education. 
- Include a mix of opportunities and challenges.
- Avoid highly technical topics. 
- Keep each topic concise.   

Output format: Return a numbered list containing:
 1. Topic title 
 2. A one-sentence description 
 """

The structured prompt gives Claude more information about what we want and what a useful response should look like.

### Comparing the Two Approaches

The difference is not simply that the second prompt is longer. The structured prompt reduces ambiguity by defining the task, context, constraints, and expected output.  

| Prompt                  | Strength                    | Limitation                                  |
|--------------------------|----------------------------|-------------------------------------------|
| Simple prompt            | Fast and flexible         | Leave more decision to the model                      |
| Structured prompt       | More explicit and predictable    | Require more effort to design             |  

To sum up ** The more predictable the output needs to be, the more explicitly the prompt should define the task and expected result.**   

In the next section, we will move from this general idea to **a concrete task: analyzing customer reviews and extracting structured information from them.**

##  Our Task: Customer Review Analysis  

We will now apply the principles from the previous section to a concrete task: analyzing customer reviews.

Our goal is to build a prompt that can:  

- determine the overall sentiment of a review  
- identify the main complaints  
- explain the sentiment classification  
- return the result in a predictable JSON structure  

Rather than writing the complete prompt at once, we will build and improve it step by step.  

### Sample Review  

Consider the following customer review:  


```text  
I recently purchased the XYZ Smartphone, and I have to say, it's been a mixed experience. On the positive side, the camera quality is absolutely stunning. The photos I've taken are crisp, clear, and the colors are vibrant. The battery life is also impressive, lasting me a full day even with heavy use.

However, there are a few issues that have been frustrating. Firstly, the phone tends to overheat when I'm using multiple apps or playing games for extended periods. It becomes uncomfortably warm to hold. Secondly, the facial recognition feature is unreliable. It often fails to recognize me in low light conditions, forcing me to use the PIN instead.

The user interface is sleek and intuitive, which I appreciate. But I've noticed occasional lag when switching between apps, which is disappointing for a phone at this price point. Customer support has been helpful in addressing some of my concerns, but they haven't been able to resolve the overheating issue.

Overall, while I'm impressed with certain aspects of the phone, the negatives are significant enough that I'm not sure I can wholeheartedly recommend it. It's a decent phone, but for the price, I expected fewer issues.  
```

This is a useful example because the review contains both positive and negative feedback. The model therefore needs to consider the review as a whole rather than simply looking for individual positive or negative phrases.

### Define the Desired Output

Before writing the prompt, we should decide what information our application actually needs.

For this task, we want three fields:  

- `sentiment_score` — the overall sentiment
- `sentiment_analysis` — a concise explanation of the classification
- `complaints` — the main issues identified in the review

For example:  

 
```json
{
  "sentiment_score": "Negative",
  "sentiment_analysis": "While the review acknowledges positive aspects (camera quality, battery life, and user interface), the overall sentiment is negative due to multiple unresolved technical issues, performance problems, and price-to-value concerns. The reviewer explicitly states reluctance to recommend the product and concludes with disappointment regarding the value proposition.",
  "complaints": [
    "Phone overheating during multiple apps usage and gaming",
    "Unreliable facial recognition system, especially in low light",
    "Performance lag when switching between apps",
    "Poor value for money considering the price point"
  ]
}
```

Defining the desired output before writing the prompt helps us turn an open-ended request into a well-defined task.

## Building the Prompt

We can now build the prompt one component at a time.

A useful structured prompt separates the task into several parts:  

```text  
Task and context   
     ↓   
    Input   
     ↓   
Analysis instructions   
     ↓   
 Output requirements

This makes the prompt easier to understand, modify, and reuse.

### Define the Task and Context  

First, describe what Claude is expected to do. This means define properly the role.

In [6]:
# Define the persona and objective clearly.
setting_the_role = """
You are an AI assistant specialized in analyzing customer reviews. 
Your task is to determine the overall sentiment of a given review 
and extract any specific complaints mentioned. 
Please follow these instructions carefully:
"""

The important part here is not the persona itself, but the clear definition of the task.

We are telling Claude what type of information it should extract and what the analysis is intended to accomplish.

### Create a Prompt Template  

Our prompt should work with different customer reviews, so we should avoid embedding a specific review directly into the prompt.

Instead, use a placeholder:

In [7]:
# Use a placeholder so the same prompt can work with different reviews
instruction_pt1 = """
1. Review the following customer feedback:

<customer_review>
{{CUSTOMER_REVIEW}}
</customer_review>
"""

The placeholder makes the prompt reusable. Thus, we can replace `{{CUSTOMER_REVIEW}}` with any review we want to analyze.

This gives us a simple template:  

```text  
Reusable prompt + Customer review  
             ↓   
        Specific request 
```  


### Define the Analysis Steps

Complex tasks can be easier to perform when the prompt clearly defines the steps that should be followed.

Instead of asking Claude to immediately produce the final answer, we can describe the analysis process:

In [8]:
# This section helps the model reason before producing the final JSON answer.
instruction_pt2 = """
2. Analyze the review using the following steps. 
Show your work in <review_breakdown> tags:

a) Key Phrase Extraction:
   - Extract and quote key phrases that indicate sentiment 
   (positive, negative, or neutral).
   - Extract and quote key phrases that suggest complaints or issues.

b) Sentiment Analysis:
   - Consider arguments for positive, negative, 
   and neutral sentiment based on the extracted phrases.
   - Determine the overall sentiment (positive, negative, or neutral) 
   based on your analysis.
   - Explain your reasoning for the sentiment classification.

c) Complaint Extraction:
   - List each specific issue or problem mentioned in the review.
   - For each complaint, provide the relevant quote from the review.
   - Count the total number of complaints found.

It's OK for this section to be quite long as you 
thoroughly break down the review.
"""

These instructions provide a consistent framework for analyzing different reviews.

Notice that we are defining the analysis process without asking the model to expose its private reasoning. The application only needs the final result.

### Define the Output Format  

The next step is to specify exactly what the application should receive.  

In [9]:
# Define the exact output schema so the answer is structured and machine-readable
instruction_pt3 = """
3. Based on your analysis, 
generate a JSON output with the following structure:

<json>
{
  "sentiment_score": "Positive|Negative|Neutral",
  "sentiment_analysis": "Explanation of sentiment classification",
  "complaints": [
    "Complaint 1",
    "Complaint 2",
    "..."
  ]
}
</json>

If no complaints are found, 
use an empty array for the "complaints" field.

Remember:
- Base your analysis solely on the content of the provided review.
- Do not make assumptions or include information 
not present in the review.
- Be objective and focus on the customer's expressed 
opinions and experiences.
- Ensure your JSON output is properly formatted 
and contains all required fields.
"""

Defining the output structure makes the expected result much clearer.

> Important: Asking a model to return JSON does not by itself guarantee valid JSON. Applications should parse and validate model-generated data before using it.

### Assemble the Prompt  

We can now combine the individual components into one reusable prompt:

In [10]:
# Combine the prompt sections into one final template
final_prompt = f"""
{setting_the_role}
{instruction_pt1}
{instruction_pt2}
{instruction_pt3}
"""

Keeping the prompt in separate components makes it easier to experiment with individual instructions.

For example, we can change the output requirements without rewriting the task description.

### Testing the Prompt

A prompt is not finished when it has been written. We need to test it with real inputs and inspect the output.

In [ ]:
#  inspect the prompt before sending it to the model
print(final_prompt)



You are an AI assistant specialized in analyzing customer reviews. 
Your task is to determine the overall sentiment of a given review 
and extract any specific complaints mentioned. 
Please follow these instructions carefully:


1. Review the following customer feedback:

<customer_review>
{{CUSTOMER_REVIEW}}
</customer_review>


2. Analyze the review using the following steps. 
Show your work in <review_breakdown> tags:

a) Key Phrase Extraction:
   - Extract and quote key phrases that indicate sentiment 
   (positive, negative, or neutral).
   - Extract and quote key phrases that suggest complaints or issues.

b) Sentiment Analysis:
   - Consider arguments for positive, negative, 
   and neutral sentiment based on the extracted phrases.
   - Determine the overall sentiment (positive, negative, or neutral) 
   based on your analysis.
   - Explain your reasoning for the sentiment classification.

c) Complaint Extraction:
   - List each specific issue or problem mentioned in the revie

## Using The Prompt

The next step is to insert a customer review and send the prompt to the model. We also parse the model response to extract only the JSON block, which is useful when the model includes extra reasoning or formatting text.

In [21]:
import re
def get_review_sentiment(review):
    #Insert the context into the prompt
    prompt = final_prompt.replace("{{CUSTOMER_REVIEW}}", review)
    # Send a request to Claude
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        messages=[
            {"role": "user", "content": prompt}        
        ]
    )
    output = response.content[0].text
    print("ENTIRE MODEL OUTPUT: ")
    print(output)
    
    sentiment = re.search(r'<json>(.*?)</json>', output, re.DOTALL)
    
    if sentiment:
        print("FINAL JSON OUTPUT: ")
        print(sentiment.group(1).strip())
    else:
        print("No sentiment analysis in the response.")

- Example review 1 

In [22]:
review1 = """
I am in love with my Acme phone.  It's incredible.
It's a little expensive, but so worth it imo.
If you can afford it, it's worth it! 
I love the colors too!
"""

In [23]:
get_review_sentiment(review1)

ENTIRE MODEL OUTPUT: 
<review_breakdown>

a) Key Phrase Extraction:

Positive Phrases:
1. "I am in love with my Acme phone"
2. "It's incredible"
3. "so worth it imo"
4. "If you can afford it, it's worth it!"
5. "I love the colors too!"

Neutral/Potentially Negative Phrases:
1. "It's a little expensive"

Complaint-suggesting Phrases:
1. "It's a little expensive" - this is the only phrase that hints at a drawback, but it's immediately softened by "but so worth it imo"

b) Sentiment Analysis:

Arguments for positive sentiment:
- Strong positive language throughout: "in love," "incredible," "worth it," "love the colors"
- Multiple affirmations of satisfaction and recommendation ("worth it" repeated twice)
- Enthusiastic tone with exclamation points
- No lingering negativity - the one potential downside (price) is immediately countered with reassurance

Arguments for negative sentiment:
- None significant found. The mention of price is not framed as a genuine complaint but rather a minor ca

- Example review 2

In [14]:
import re

def get_review_sentiment(review):
    prompt = final_prompt.replace("{{CUSTOMER_REVIEW}}", review)

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    # Some models return thinking blocks before the final text block.
    text_blocks = [
        block for block in response.content
        if getattr(block, "type", None) == "text"
    ]

    if not text_blocks:
        print("No text output returned by the model.")
        return

    output = "\n".join(block.text for block in text_blocks)
    print("ENTIRE MODEL OUTPUT:")
    print(output)

    sentiment = re.search(r"<json>(.*?)</json>", output, re.DOTALL)
    if sentiment:
        print("FINAL JSON OUTPUT:")
        print(sentiment.group(1).strip())
    else:
        print("No sentiment analysis in the response.")

In [15]:
review2 = """
I recently bought the ABC Smartwatch, and I have to say, it's been a 
bit of a rollercoaster. Let's start with the good stuff - 
the design is absolutely gorgeous. It's sleek, lightweight, 
and looks fantastic on my wrist. The display is crisp and bright, 
even in direct sunlight. I also love how customizable the watch 
faces are, allowing me to switch up the look whenever I want.
However, there are some significant downsides that I can't ignore. 
The battery life is terrible - I'm lucky if I get through a full 
day without needing to charge it. This is especially frustrating
when I'm traveling or out for long periods. The fitness tracking 
features, which were a big selling point for me, are hit or miss. 
It often doesn't accurately count my steps or calculate calories burned 
during workouts. Another issue I've encountered is with the touch 
sensitivity. Sometimes it's overly sensitive, registering accidental 
touches, while other times I have to tap multiple times for it to 
respond. It's inconsistent and annoying, especially when I'm trying to 
quickly check notifications or start a workout.On the plus side, the 
integration with my smartphone is seamless, and I appreciate being able 
to respond to texts and calls from my wrist. The water resistance has 
also held up well - I've worn it while swimming without any problems.
Customer service has been decent. They were quick to respond when I 
reported the battery issue, but their solution of turning off certain 
features defeats the purpose of having a smartwatch. All in all, while 
there are aspects of the ABC Smartwatch that I really like, the battery 
life and inconsistent performance are major drawbacks. For the price 
point, I expected better. It's not worth your money.
"""

In [16]:
get_review_sentiment(review2)

ENTIRE MODEL OUTPUT:
<review_breakdown>

**a) Key Phrase Extraction:**

*Positive/Neutral sentiment phrases:*
- "the design is absolutely gorgeous"
- "It's sleek, lightweight, and looks fantastic on my wrist"
- "The display is crisp and bright, even in direct sunlight"
- "I also love how customizable the watch faces are"
- "the integration with my smartphone is seamless"
- "I appreciate being able to respond to texts and calls from my wrist"
- "The water resistance has also held up well"
- "Customer service has been decent"
- "They were quick to respond when I reported the battery issue"

*Negative sentiment/complaint phrases:*
- "it's been a bit of a rollercoaster"
- "there are some significant downsides that I can't ignore"
- "The battery life is terrible"
- "I'm lucky if I get through a full day without needing to charge it"
- "especially frustrating when I'm traveling or out for long periods"
- "The fitness tracking features...are hit or miss"
- "It often doesn't accurately count m

## Examples

Examples are especially useful in prompt engineering because they show the model exactly what a good answer should look like. A well-crafted few-shot example can reduce ambiguity and improve consistency in output formatting.

In [17]:
examples = """
Here are some example inputs and outputs to help you understand
the sort of analysis we're looking for
<examples>
<example1>

<customer_review>
I recently bought the ABC Smartwatch, and I have to say, it's been a bit of a rollercoaster. Let's start with the good stuff - the design is absolutely gorgeous. It's sleek, lightweight, and looks fantastic on my wrist. The display is crisp and bright, even in direct sunlight. I also love how customizable the watch faces are, allowing me to switch up the look whenever I want.
However, there are some significant downsides that I can't ignore. The battery life is terrible - I'm lucky if I get through a full day without needing to charge it. This is especially frustrating when I'm traveling or out for long periods. The fitness tracking features, which were a big selling point for me, are hit or miss. It often doesn't accurately count my steps or calculate calories burned during workouts.
Another issue I've encountered is with the touch sensitivity. Sometimes it's overly sensitive, registering accidental touches, while other times I have to tap multiple times for it to respond. It's inconsistent and annoying, especially when I'm trying to quickly check notifications or start a workout.
On the plus side, the integration with my smartphone is seamless, and I appreciate being able to respond to texts and calls from my wrist. The water resistance has also held up well - I've worn it while swimming without any problems.
Customer service has been decent. They were quick to respond when I reported the battery issue, but their solution of turning off certain features defeats the purpose of having a smartwatch.
All in all, while there are aspects of the ABC Smartwatch that I really like, the battery life and inconsistent performance are major drawbacks. For the price point, I expected better. It's not worth your money.
</customer_review>

<ideal_output>
<review_analysis>
a) Key Phrase Extraction:

Positive Phrases:
1. "design is absolutely gorgeous"
2. "sleek, lightweight, and looks fantastic"
3. "display is crisp and bright"
4. "love how customizable the watch faces are"
5. "integration with my smartphone is seamless"
6. "water resistance has held up well"
7. "customer service has been decent"

Negative Phrases:
1. "bit of a rollercoaster"
2. "battery life is terrible"
3. "lucky if I get through a full day"
4. "fitness tracking features are hit or miss"
5. "doesn't accurately count my steps"
6. "touch sensitivity... overly sensitive"
7. "have to tap multiple times"
8. "inconsistent and annoying"
9. "solution of turning off certain features defeats the purpose"
10. "battery life and inconsistent performance are major drawbacks"
11. "not worth your money"

b) Sentiment Analysis:
Arguments for positive sentiment:
- The design and aesthetics of the watch are highly praised
- The display quality is commended
- Customization options are appreciated
- Smartphone integration works well
- Water resistance feature is effective
- Customer service responded quickly

Arguments for negative sentiment:
- Battery life is severely criticized
- Fitness tracking, a key feature, is unreliable
- Touch sensitivity issues cause frustration
- The proposed solution compromises functionality
- The price doesn't justify the performance
- The reviewer explicitly states it's not worth the money

Considering the strength and frequency of sentiments:
- Positive sentiments are mostly about aesthetics and basic functions
- Negative sentiments relate to core functionalities and user experience
- The reviewer uses stronger language for negative aspects ("terrible", "major drawbacks", "not worth your money")
- The conclusion emphasizes disappointment and recommends against purchase

Based on this analysis, the overall sentiment leans negative. While the reviewer appreciates certain aspects, the core issues with battery life, performance, and value for money overshadow the positives. The explicit recommendation against purchase is a strong indicator of overall negative sentiment.

c) Complaint Extraction:
1. Poor battery life
   Quote: "The battery life is terrible - I'm lucky if I get through a full day without needing to charge it"

2. Inaccurate fitness tracking
   Quote: "The fitness tracking features, which were a big selling point for me, are hit or miss. It often doesn't accurately count my steps or calculate calories burned during workouts"

3. Inconsistent touch sensitivity
   Quote: "Sometimes it's overly sensitive, registering accidental touches, while other times I have to tap multiple times for it to respond"

4. Inadequate solution to problems
   Quote: "their solution of turning off certain features defeats the purpose of having a smartwatch"

Total Complaints: 4

d) Summary:
The analysis of the ABC Smartwatch review reveals a predominantly negative sentiment. While the customer appreciates the design, display quality, and some basic features, these positives are outweighed by significant issues with core functionalities. The main complaints center around poor battery life, inaccurate fitness tracking, inconsistent touch sensitivity, and inadequate problem-solving from customer service. The reviewer concludes that the product is not worth the money, emphasizing that the drawbacks are too significant to justify the price point.
</review_analysis>

<json>
{
  "sentiment_score": "Negative",
  "sentiment_analysis": "While the review acknowledges positive aspects like design and basic functionality, the overall sentiment is negative due to significant issues with core features (battery life, fitness tracking, touch sensitivity). The reviewer explicitly states it's 'not worth your money' and emphasizes major drawbacks that overshadow the positive elements.",
  "complaints": [
    "Poor battery life requiring daily charging",
    "Inaccurate fitness tracking features",
    "Inconsistent touch sensitivity causing user interface problems",
    "Inadequate customer service solutions that compromise device functionality"
  ]
}
</json>
</ideal_output>

</example1>
</examples>
"""

In [18]:
final_prompt = f"""
{setting_the_role}
{instruction_pt1}
{instruction_pt2}
{instruction_pt3}

<examples>
{examples}
</examples>
"""

In [19]:
import re

def get_review_sentiment(review):
    prompt = final_prompt.replace("{{CUSTOMER_REVIEW}}", review)

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    # Some models return thinking blocks before the final text block.
    text_blocks = [
        block for block in response.content
        if getattr(block, "type", None) == "text"
    ]

    if not text_blocks:
        print("No text output returned by the model.")
        return

    output = "\n".join(block.text for block in text_blocks)
    print("ENTIRE MODEL OUTPUT:")
    print(output)

    sentiment = re.search(r"<json>(.*?)</json>", output, re.DOTALL)
    if sentiment:
        print("FINAL JSON OUTPUT:")
        print(sentiment.group(1).strip())
    else:
        print("No sentiment analysis in the response.")

In [20]:
get_review_sentiment(review2)

ENTIRE MODEL OUTPUT:
<review_breakdown>

**a) Key Phrase Extraction:**

*Positive/Neutral Phrases:*
1. "the design is absolutely gorgeous"
2. "sleek, lightweight, and looks fantastic on my wrist"
3. "The display is crisp and bright, even in direct sunlight"
4. "I also love how customizable the watch faces are"
5. "the integration with my smartphone is seamless"
6. "I appreciate being able to respond to texts and calls from my wrist"
7. "The water resistance has also held up well"
8. "I've worn it while swimming without any problems"
9. "Customer service has been decent"
10. "They were quick to respond when I reported the battery issue"

*Negative/Complaint Phrases:*
1. "it's been a bit of a rollercoaster"
2. "The battery life is terrible"
3. "I'm lucky if I get through a full day without needing to charge it"
4. "especially frustrating when I'm traveling or out for long periods"
5. "The fitness tracking features... are hit or miss"
6. "It often doesn't accurately count my steps or calc

## Conclusion  

In this notebook, we moved from a simple prompt to a structured, reusable prompt for a real-world task, demostrating that the quality of the output depends not only on the model, but also on how clearly the task is framed.

Briefly,
- a prompt should clearly define the task and provide relevant context.
- Separating instructions from input data makes prompts easier to reuse.  
- Defining the desired output helps make model responses more predictable.  
- Complex tasks can benefit from clearly defined analysis steps.  
- Examples can demonstrate the expected behavior and output format.  
- Generated structured data should be parsed and validated before being used by an application.  
- A prompt should be tested with multiple inputs rather than a single example.  
- Prompt quality should be evaluated against the requirements of the task.  

The overall workflow is:  

```  
Define the task
      ↓
Define the desired output
      ↓
Build the prompt
      ↓
Test the prompt
      ↓
Improve with examples
      ↓
Evaluate across multiple inputs
      ↓
Iterate  
```

The goal of prompt engineering, or called also prompt design, is not simply to write longer prompts. It is to provide (clarify) the right instructions, context, examples, and constraints needed to make the model useful for a specific task. Good prompting is iterative, measurable, and grounded in the task you want to automate. 

## Extra practice    

The following challenges extend the customer-review example and encourage you to apply the same prompt-engineering process to new situations.

### Challenge 1: Change the Domain  

Adapt the sentiment-analysis prompt to a different domain.  

Choose one:  

- hotel reviews  
- SaaS product feedback  
- restaurant reviews  
- social media comments  

Modify the prompt so that it analyzes the new type of input while preserving the overall workflow.  

In [ ]:
new_review = """ 
Replace this text with a review from your chosen domain. 
""" 
result = get_review_sentiment(new_review) 
print(result)

In [ ]:
# Extra Practice: test the prompt on a new domain
import json
import re

def analyze_review(review_text, prompt_template):
    prompt = prompt_template.replace("{{CUSTOMER_REVIEW}}", review_text)
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}]
    )

    text_blocks = [
        block.text for block in response.content
        if getattr(block, "type", None) == "text"
    ]

    output = "\n".join(text_blocks)
    match = re.search(r"<json>(.*?)</json>", output, re.DOTALL)

    if match:
        return json.loads(match.group(1).strip())
    return {"error": "No JSON output found"}

# Example review from another domain
hotel_review = """
The hotel was beautiful and the staff were friendly,
but the room was noisy at night and the Wi-Fi was unreliable.
The breakfast was good, but the check-in process took too long.
Overall, I liked the location and design, but the service issues
made the stay less enjoyable.
"""

result = analyze_review(hotel_review, final_prompt)
print(result)

### Challenge 2: Extend the Schema  

Extend the output schema with additional fields.  

For example:

- `customer_emotion`
- `urgency`
- `recommended_action`

A possible structure is:

In [ ]:
extended_prompt = """
Analyze the customer review and return JSON with: 

{ "sentiment_score": "Positive | Negative | Neutral", 
"customer_emotion": "...", "urgency": "Low | Medium | High",
"recommended_action": "...", "complaints": [] } 
"""

Test the extended schema with several reviews and consider whether each new field provides useful information.

In [ ]:
import json
import re

def analyze_review_with_extended_schema(review_text, prompt_template):
    prompt = prompt_template.replace("{{CUSTOMER_REVIEW}}", review_text)

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    text_blocks = [
        block.text for block in response.content
        if getattr(block, "type", None) == "text"
    ]

    output = "\n".join(text_blocks)
    match = re.search(r"<json>(.*?)</json>", output, re.DOTALL)

    if not match:
        return {"error": "No valid JSON found in the model response."}

    return json.loads(match.group(1).strip())


# Example prompt template with extended schema
extended_prompt = """
You are an AI assistant specialized in analyzing customer reviews.

Review the following customer feedback:

<customer_review>
{{CUSTOMER_REVIEW}}
</customer_review>

Extract:
- sentiment_score: Positive | Negative | Neutral
- customer_emotion: a short emotion label
- urgency: Low | Medium | High
- recommended_action: a short action recommendation
- complaints: a list of issues mentioned in the review

Return valid JSON only in the following format:
<json>
{
  "sentiment_score": "Positive|Negative|Neutral",
  "customer_emotion": "short description",
  "urgency": "Low|Medium|High",
  "recommended_action": "short recommendation",
  "complaints": ["issue 1", "issue 2"]
}
</json>
"""

# Sample reviews to compare model behavior
reviews = [
    "The app is very easy to use, but the loading speed is slow and it crashes sometimes.",
    "I love the product design and customer support, but the battery drains too fast.",
    "The service was terrible and I would not recommend this product to anyone."
]

for review in reviews:
    result = analyze_review_with_extended_schema(review, extended_prompt)
    print("Review:", review)
    print(json.dumps(result, indent=2))
    print("-" * 80)

### Challenge 3: Test Edge Cases  

Test your prompt with reviews that may be more difficult to classify.

For example:

In [ ]:
edge_case_reviews = [ 
    "It's not bad, but I wouldn't buy it again.", 
    "The product is amazing! Unfortunately, it stopped working after two days.",
    "The design is great. Everything else is disappointing.", 
    "It works exactly as expected." 
    ] 
for review in edge_case_reviews: 
    output = get_review_sentiment(review) 
    result = parse_json_output(output) 
    print("Review:", review) 
    print(json.dumps(result, indent=2)) 
    print("-" * 80)

Pay particular attention to mixed sentiment,  contradictory statements, 
very short reviews, ambiguous language.  

### Challenge 4: Evaluate the Prompt  

Create a small evaluation table for your test cases.

For each response, record whether:

- the sentiment is correct  
- the main complaints were identified  
- the output contains the required fields  
- the JSON is valid  
- the result is consistent with the other test cases  

For example:

In [ ]:
evaluation = [ 
    {
        "review": "Review 1", 
        "sentiment_correct": True, 
        "complaints_complete": True, 
        "valid_json": True 
        }, 
        { "review": "Review 2",
          "sentiment_correct": False, 
          "complaints_complete": True, 
          "valid_json": True 
          } 
    ] 
for result in evaluation: 
    print(result)

Use the evaluation results to identify weaknesses in your prompt and make another iteration.